In [1]:
import sys
import base64
import os

import pandas as pd
from pathlib import Path
from openai import Client
from pypdf import PdfReader
import yagmail, requests, json, tempfile, datetime

def pdf_to_string(pdf_path):
    #Converts a pdf file in the same folder into a string
    pdf = PdfReader(pdf_path)
    text = ""

    for page in pdf.pages:
        t = page.extract_text()

        if t:
            text += t + "\n"

    return text

def compose(company, data):
    #Asks ChatGPT to generate an email that is an analytics report for the current company
    global ai
    print("Composing Email...")
    response = ai.chat.completions.create(
        model="gpt-5.6-luna", #This is the ChatGPT model used, all the models should do so well, modify this if you want price or quality to improve
        messages=[
            {"role": "system", "content": f"You are a financial assistant that anylizes the trucking income and expense data of my company based on the provided data alone. You are writing in email format and use the corporate we/us/our when refering to yourself. You are writing this email from the comapny Duke.ai and will not sign off using a name and position, just the company name. This email is to be sent to my company {company} and will not include a subject line. Please translate all your emails into html without changing the email at all."},
            #This above is the prompt, it took a while to refine, please don't change unless the email comes out wrong
            {"role": "assistant", "content": data},
            {"role": "user", "content": "write an analytics report of my company's expenses based on the data provided"}
            #This above tells the AI what it is, if the email comes out wrong, don't change this, change the prompt at the top
        ]
    )
    return response.choices[0].message.content[7:-4]

def email(name, data, e):
    #sends and email to the selected company and email address with an AI email generated by ChatGPT
    global subject, yag

    print("Emailing Report...")

    yag.send(
        to=e,
        subject=subject,
        contents=compose(name, data)
    )

def genReport():
    #Posts to the API, telling the API we want an Expenses report
    global cust_id, headers_post, payload

    print("Generating Report...")

    url = f"https://m4rvl2hkdh.execute-api.us-east-1.amazonaws.com/api/{cust_id}/gen_report"
    try:
        response = requests.post(url, headers=headers_post, json=payload, timeout=30)

        # print(f"Status Code: {response.status_code}")

        # # Try to print JSON response; fall back to raw text if not JSON
        # try:
        #     print("Response Body:")
        #     print(json.dumps(response.json(), indent=2))
        # except ValueError:
        #     print("Response Body (raw):")
        #     print(response.text)

        return response

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

def downloadReport():
    #Downloads the report into the folder this program is in using a Get request on the API
    global cust_id, headers_get, payload

    print("Downloading Report...")

    url = f"https://25g071tuih.execute-api.us-east-1.amazonaws.com/api/{cust_id}/Expense.pdf"

    response = requests.get(url, headers=headers_get, timeout=30, stream=True)
    match response.status_code:
        case 403:
            print("Error: customer id likely incorrect\nCheck customer id for typos")
            return
        case 200:
            pass
        case _:
            print("Error: API issue.\nCheck authentication, headers, and code for inconsistencies.")
            return
    # print(response.status_code)
    # print(response.headers.get("Content-Type"))
    # print(response.content[:200])

    pdf_bytes = base64.b64decode(response.text)
    # print(pdf_bytes)

    with open("file.pdf", "wb") as f:
        f.write(pdf_bytes)
        f.close()

    # with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
    #     f.write(report.content)
    #     filename = f.name
    #     f.close()
    #     print(filename)

def reportsFromExcel():
    #Goes through each email in the excel spreadsheet in this program's folder and emails them a report
    global cust_id
    path = Path(os.getcwd())
    print(f"Current Path: {path}")

    files = []

    for file in path.iterdir():
        print()
        if file.is_file():
            if  file.suffix == ".xlsx":
                files.append(file.name)

    if len(files) > 1:
        input("Error: Multiple .xlsx files detected. \nConsolidate multiple files or remove unnecessary duplicates before rerunning this program.")
        return
    if len(files) < 1:
        input("Error: File not found.\nMove .xslx spreadsheet into the same folder as this program.")
        return

    df = pd.read_excel(files[0])

    for e, c in df["email"], df["company name"]:
        cust_id = e
        print("------------------------------")
        print(f"Cust_id: {cust_id}")
        genReport()
        downloadReport()
        email(c, pdf_to_string("file.pdf"), cust_id)
    print("------------------------------")
    input("Report generation complete.")


#AI Setup
subject = "Monthly Expenses Analysis" #The subject of the emails
yag = yagmail.Client('email', 'App password')
#TODO: Set up 2FA on the email you wish to send automated emails from, paste the email address in the first string, then generate an "App password" and paste that in the second string

apiKey = "your AI key here"
#TODO: Copy your own ChatGPT API key here

ai = Client(api_key=apiKey)

#API setup
time = datetime.datetime.now()
payload = {
    "from": time.strftime("%Y-01-01"),       # Report start date (YYYY-MM-DD)
    "to": time.strftime("%Y-%m-%d"),         # Report end date (YYYY-MM-DD)
    "report_type": "Expense",   # Type of report to generate
    "file_type": "pdf",         # Output file format
}

token = "your duke.ai API key here"
#TODO: Paste your own personal API key here for the program to run properly

headers_post = {#Headers for posting the report request
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}",
}

headers_get = {#Headers for getting the pdf from the API
    "Content-Type": "application/octet-stream",
    "Authorization": f"Bearer {token}"
}

cust_id = "" #No need to modify this, the customer id is read from the spreadsheet, this is just to define it at the start so it can be read by the other functions

reportsFromExcel()

{'from': '2026-01-01', 'to': '2026-08-19', 'report_type': 'Expense', 'file_type': 'pdf'}
